In [1]:
%run nb_helpers.py

from datar.data import iris
from datar.all import *

nb_header(pack, unpack)

### <div style="background-color: #EEE; padding: 5px 0 8px 0">★ pack</div>

##### Makes df narrow by collapsing a set of columns into a single df-column.

##### Args:
&emsp;&emsp;`_data`: A data frame  
&emsp;&emsp;`**cols`: Columns to pack  
&emsp;&emsp;`_names_sep`: If `None`, the default, the names will be left as is.  
&emsp;&emsp;&emsp;&emsp;Inner names will come from the former outer names  
&emsp;&emsp;&emsp;&emsp;If a string, the inner and outer names will be used together.  
&emsp;&emsp;&emsp;&emsp;The names of the new outer columns will be formed by pasting  
&emsp;&emsp;&emsp;&emsp;together the outer and the inner column names, separated by  
&emsp;&emsp;&emsp;&emsp;`_names_sep`.  


### <div style="background-color: #EEE; padding: 5px 0 8px 0">★ unpack</div>

##### Makes df wider by expanding df-columns back out into individual columns.

For empty columns, the column is kept asis, instead of removing it.  

##### Args:
&emsp;&emsp;`data`: A data frame  
&emsp;&emsp;`cols`: Columns to unpack  
&emsp;&emsp;`names_sep`: If `None`, the default, the names will be left as is.  
&emsp;&emsp;&emsp;&emsp;Inner names will come from the former outer names  
&emsp;&emsp;&emsp;&emsp;If a string, the inner and outer names will be used together.  
&emsp;&emsp;&emsp;&emsp;The names of the new outer columns will be formed by pasting  
&emsp;&emsp;&emsp;&emsp;together the outer and the inner column names, separated by  
&emsp;&emsp;&emsp;&emsp;`_names_sep`.  

&emsp;&emsp;`name_repair`: treatment of problematic column names:  
&emsp;&emsp;&emsp;&emsp;- "minimal": No name repair or checks, beyond basic existence,  

&emsp;&emsp;&emsp;&emsp;- "unique": Make sure names are unique and not empty,  

&emsp;&emsp;&emsp;&emsp;- "check_unique": (default value), no name repair,  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;but check they are unique,  

&emsp;&emsp;&emsp;&emsp;- "universal": Make the names unique and syntactic  

&emsp;&emsp;&emsp;&emsp;- a function: apply custom name repair  

##### Returns:
&emsp;&emsp;Data frame with given columns unpacked.  


In [2]:
df = tibble(x1 = c[1:2], x2 = c[4:5], x3 = c[7:8], y = c[1:2])
df

x1,x2,x3,y
i64,i64,i64,i64
1,4,7,1
2,5,8,2


In [3]:
df >> pack(x=starts_with('x'))

y,x
i64,struct[3]
1,"{1,4,7}"
2,"{2,5,8}"


In [4]:
df >> pack(x=c(f.x1, f.x2, f.x3), y=f.y)

y,x
struct[1],struct[3]
{1},"{1,4,7}"
{2},"{2,5,8}"


In [5]:
iris >> pack(
    Sepal=starts_with("Sepal"),
    Petal=starts_with("Petal"),
    _names_sep="_"
)

Species,Sepal,Petal
str,struct[2],struct[2]
"""setosa""","{5.1,3.5}","{1.4,0.2}"
"""setosa""","{4.9,3.0}","{1.4,0.2}"
"""setosa""","{4.7,3.2}","{1.3,0.2}"
"""setosa""","{4.6,3.1}","{1.5,0.2}"
"""setosa""","{5.0,3.6}","{1.4,0.2}"
…,…,…
"""virginica""","{6.7,3.0}","{5.2,2.3}"
"""virginica""","{6.3,2.5}","{5.0,1.9}"
"""virginica""","{6.5,3.0}","{5.2,2.0}"


In [6]:
# Unpacking ===========================================================

df = tibble(
  x = c[1:3],
  y = tibble(a = c[1:3], b = c[3:1]),
  z = tibble(X = c("a", "b", "c"), Y = runif(3), Z = c(TRUE, FALSE, NA))
)
df

x,y,z
i64,struct[2],struct[3]
1,"{1,3}","{""a"",0.964516,true}"
2,"{2,2}","{""b"",0.980663,false}"
3,"{3,1}","{""c"",0.746063,null}"


In [7]:
df >> unpack(f.y)

x,a,b,z
i64,i64,i64,struct[3]
1,1,3,"{""a"",0.964516,true}"
2,2,2,"{""b"",0.980663,false}"
3,3,1,"{""c"",0.746063,null}"


In [8]:
df >> unpack(c(f.y, f.z))

x,a,b,X,Y,Z
i64,i64,i64,str,f64,bool
1,1,3,"""a""",0.964516,true
2,2,2,"""b""",0.980663,false
3,3,1,"""c""",0.746063,null


In [9]:
df >> unpack(c(f.y, f.z), names_sep="_")


x,a,b,X,Y,Z
i64,i64,i64,str,f64,bool
1,1,3,"""a""",0.964516,true
2,2,2,"""b""",0.980663,false
3,3,1,"""c""",0.746063,null


In [10]:
with try_catch():
    # indexes from inner data frame counts
    df >> unpack(c(2,3))

[IndexError] list index out of range


In [12]:
# df >> unpack(c(2,4))